In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:13pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:80px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:2px;}
table.dataframe{font-size:5xpt;} 
</style>
"""))

**<font size="6" color="red">ch4.RNN(Recurrent Neural Network)**</font>

- 순서나 시간 데이터가 중요할 때 ex.번역, 음성인식, 주가 예측
# 1. 문맥을 이용하여 모델 만들기

In [2]:
text = """ 경마장에 있는 말이 뛰고 있다.
그의 말이 법이다
가는 말이 고와야 오는 말이 곱다"""
text1 = "겨울이 오는 날"

In [4]:
from keras_preprocessing.text import Tokenizer
t = Tokenizer()
t.fit_on_texts([text, text1])
encoded = t.texts_to_sequences([text, text1])
print(encoded)
print(t.word_index)

[[3, 4, 1, 5, 6, 7, 1, 8, 9, 1, 10, 2, 1, 11], [12, 2, 13]]
{'말이': 1, '오는': 2, '경마장에': 3, '있는': 4, '뛰고': 5, '있다': 6, '그의': 7, '법이다': 8, '가는': 9, '고와야': 10, '곱다': 11, '겨울이': 12, '날': 13}


In [5]:
text = """ 경마장에 있는 말이 뛰고 있다.
그의 말이 법이다
가는 말이 고와야 오는 말이 곱다"""

In [6]:
# 문자를 인덱스 시퀀스로 변환하기 위한 과정
t = Tokenizer()
t.fit_on_texts([text])
print(t.word_index)

{'말이': 1, '경마장에': 2, '있는': 3, '뛰고': 4, '있다': 5, '그의': 6, '법이다': 7, '가는': 8, '고와야': 9, '오는': 10, '곱다': 11}


In [10]:
# 문자열 리스트를 인덱스 시퀀스로 변환
print(t.texts_to_sequences(['경마장에 말이 있다', '말이 뛴다']))
print(t.texts_to_sequences(['가는 말이 곱다'])[0])

[[2, 1, 5], [1]]
[8, 1, 11]


In [11]:
for key, value in t.word_index.items():
    print(key, value)

말이 1
경마장에 2
있는 3
뛰고 4
있다 5
그의 6
법이다 7
가는 8
고와야 9
오는 10
곱다 11


In [ ]:
text = """ 경마장에 있는 말이 뛰고 있다.
그의 말이 법이다
가는 말이 고와야 오는 말이 곱다"""

In [16]:
# text를 학습시키기 위한 ['경마장에 있는', '경마장에 있는 말이', ....]
sequences = []
for line in text.split('\n'):
    print('원 문장 : ', line)
    encoded = t.texts_to_sequences([line])[0]
    print('encoded된 문장 :', encoded)
    for i in range(0, len(encoded)-1): # 시각 index
        for j in range(i+2, len(encoded)+1): # 끝나는 index
            sequences.append(encoded[i:j])
print('sequences와 해설 출력')
for sequence in sequences:
#      print(sequence)
    for word_seq in sequence:
        for key, value in t.word_index.items():
            if word_seq==value:
                print("{}:{}".format(word_seq,key), end=' ')
                break
    print()

원 문장 :   경마장에 있는 말이 뛰고 있다.
encoded된 문장 : [2, 3, 1, 4, 5]
원 문장 :  그의 말이 법이다
encoded된 문장 : [6, 1, 7]
원 문장 :  가는 말이 고와야 오는 말이 곱다
encoded된 문장 : [8, 1, 9, 10, 1, 11]
sequences와 해설 출력
2:경마장에 3:있는 
2:경마장에 3:있는 1:말이 
2:경마장에 3:있는 1:말이 4:뛰고 
2:경마장에 3:있는 1:말이 4:뛰고 5:있다 
3:있는 1:말이 
3:있는 1:말이 4:뛰고 
3:있는 1:말이 4:뛰고 5:있다 
1:말이 4:뛰고 
1:말이 4:뛰고 5:있다 
4:뛰고 5:있다 
6:그의 1:말이 
6:그의 1:말이 7:법이다 
1:말이 7:법이다 
8:가는 1:말이 
8:가는 1:말이 9:고와야 
8:가는 1:말이 9:고와야 10:오는 
8:가는 1:말이 9:고와야 10:오는 1:말이 
8:가는 1:말이 9:고와야 10:오는 1:말이 11:곱다 
1:말이 9:고와야 
1:말이 9:고와야 10:오는 
1:말이 9:고와야 10:오는 1:말이 
1:말이 9:고와야 10:오는 1:말이 11:곱다 
9:고와야 10:오는 
9:고와야 10:오는 1:말이 
9:고와야 10:오는 1:말이 11:곱다 
10:오는 1:말이 
10:오는 1:말이 11:곱다 
1:말이 11:곱다 


In [18]:
# sequences의 길이를 모두 길게(padding
my_len = max([len(seq) for seq in sequences])
my_len

6

In [ ]:
# 

In [20]:
# 독립변수(X)와 타겟변수(y)를 분리
X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]

#단어 갯수
vocab_size = len(t.word_index)
vocab_size

NameError: name 'padded_sequences' is not defined

In [19]:
# 종속변수의 원핫인코딩
from tensorflow.keras.utils import to_categorical
Y = to_categorical(y, vocab_size+1)
Y

NameError: name 'y' is not defined

In [ ]:
# 모델 생성(Embedding → RNN → Dense층)
# Embedding층의 입력은 12(희소행렬) → 출력은 10. 이 단계에서 필요한 Embedding weight matrix
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
#교안 106p
model = Sequential(name='sequential')
model.add(Embedding(input_dim=vocab_size+1,
                   output_dim=10,
                   input_lenth))

In [21]:
# 학습과정 설정
model.compile(loss='categorical_crossentropy',
             optimizer='adam',
             metrics=['accuracy'])
# 학습시키기
hist = model.fit(X, Y, epoches=300, verbose=2)

NameError: name 'model' is not defined

In [ ]:
# 학습과정 살펴보기(시각화)
import matplotlib.pyplot as plt
fig, loss_ax = plt.subplots(figsize=(15,6))
loss_ax.plot(hist.history['loss'], 'y', label='train_loss')
acc_ax = loss_ax.twinx()
acc_ax.plot(hist.history['accuracy'], 'g', label='train_accuracy')

acc_ax.set_ylabel('accuracy')
loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss')
loss_ax.legend(bbox_to_anchor=(0.97,0.75))
acc_ax.legend(loc='center right')
plt.show()

In [23]:
# 모델 사용하기 (경마장에 있는 → 말이)
from tensorflow.keras.preprocessing.sequence import pad_sequences
encoded = t.text_to_sequences(['경마장에 있는'])[0]
input_data = pad_sequences([encoded], maxlen=5, padding='pre')
print('모델에 들어갈 입력 데이터 :', input_data)
result = model.predict(input_data).argmax(axis=1)
print('모델 예측 결과 :', result)
for key, value in t.word_index.items():
    if value==result:
        print('가장 확률이 높은 단어(예측된 단어) :', key)
        break

SyntaxError: incomplete input (1746097756.py, line 7)

# 2. 다음 문맥 예측해보기

In [26]:
# '가는 말이' 이후에 올 단어 4개 예측
def sentence_generation(model, t, current_word, n):
    print('입력된 단어 :', current_word)
    for i in range(1, n+1):
        encoded = t.text_to_sequences([current_word])[0]
        input_data = pad_sequences([encoded], maxlen=5, padding='pre')
        result = model.predict(input_data, verbose=0).argmax(axis=1)
        for key, value in t.word_index.items():
            if value==result:
                print(f'{i}번째 {key}:{result}')
                current_word = current_word + ' ' + key
                break
    return current_word

In [25]:
sentence_generation(model, t, '가는 말이', 4)

NameError: name 'model' is not defined